# Avaliacao Experimental Sistematica - CNN Multissatelite

Notebook de experimentacao controlada para comparar configuracoes de CNN,
com execucao por satelite (um por vez) para evitar estouro de RAM:
1. Sentinel-2
2. Landsat 8/9
3. MODIS

Itens cobertos:
1. Protocolo experimental
2. Multiplos experimentos
3. Analise comparativa
4. Estudo de impacto/ablacao
5. Interpretacao critica


## Reuso de funcoes do `src/`

Funcoes reaproveitadas diretamente:
- `src.models.cnn_builder.build_cnn_model`
- `src.models.cnn_builder.get_model_architecture_summary`
- `src.models.cnn_tf_data_pipeline.build_train_val_tf_data`
- `src.models.cnn_tf_data_pipeline.build_tf_data_pipeline`
- `src.models.cnn_data_prep.labels_from_extracted_codes`
- `src.train.callbacks.get_training_callbacks`
- `src.utils.metrics.classification_metrics_extended`
- `src.models.cnn_config.list_available_configs` (referencia de configs)


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import json
import gc
import math
import random
import time
import copy
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

try:
    import pyarrow.parquet as pq
except Exception:
    pq = None

try:
    import tensorflow as tf
except Exception as exc:
    raise ImportError('TensorFlow nao esta instalado. Use: pip install tensorflow') from exc

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists() and (repo_root.parent / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.models.cnn_builder import build_cnn_model, get_model_architecture_summary
from src.models.cnn_tf_data_pipeline import build_train_val_tf_data, build_tf_data_pipeline
from src.models.cnn_data_prep import labels_from_extracted_codes
from src.train.callbacks import get_training_callbacks
from src.utils.metrics import classification_metrics_extended
from src.models.cnn_config import list_available_configs

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (11, 5)

print('Repo root:', repo_root)
print('TensorFlow:', tf.__version__)
print('Configs YAML ja existentes em src/models/configs:', list_available_configs())


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_ROOT = repo_root / 'outputs' / 'a06_cnn_experimentos'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CODES_PATH = repo_root / 'data' / 'extracted_codes.json'
MLP_BASELINE_PATH = repo_root / 'outputs' / 'a03_mlp_baseline' / 'mlp_baseline_metrics.json'

SENSOR_ORDER = ['sentinel2', 'landsat89', 'modis']
SENSORS = {
    'sentinel2': {
        'display_name': 'Sentinel-2',
        'parquet_path': repo_root / 'data' / 'parquet' / 'sentinel_completo.parquet',
    },
    'landsat89': {
        'display_name': 'Landsat 8/9',
        'parquet_path': repo_root / 'data' / 'parquet' / 'landsat89_completo.parquet',
    },
    'modis': {
        'display_name': 'MODIS',
        'parquet_path': repo_root / 'data' / 'parquet' / 'modis_completo.parquet',
    },
}

DATA_CFG = {
    'image_col': 'image_id',
    'pixel_col': 'pixel_idx',
    'band_prefix': 'B',
    'max_images_per_sensor': None,
    'min_images_per_class': 20,
    'train_size': 0.70,
    'val_size': 0.15,
    'test_size': 0.15,
}

BASE_MODEL_CFG = {
    'conv1_filters': 32,
    'conv2_filters': 64,
    'kernel_size': (3, 3),
    'dense_units': 128,
    'dropout_rate': 0.5,
    'conv_dropout_rate': 0.2,
    'l2_regularizer': 1e-3,
}

BASE_TRAIN_CFG = {
    'epochs': 35,
    'batch_size': 32,
    'learning_rate': 1e-3,
    'normalization': 'zscore',
    'augment_train': True,
    'patience': 8,
    'min_delta': 1e-4,
    'reduce_lr_patience': 4,
    'reduce_lr_factor': 0.5,
    'min_lr': 1e-6,
}

EXPERIMENT_PLAN = [
    {'name': 'baseline', 'model_overrides': {}, 'train_overrides': {}},
    {'name': 'no_augmentation', 'model_overrides': {}, 'train_overrides': {'augment_train': False}},
    {'name': 'no_l2', 'model_overrides': {'l2_regularizer': 0.0}, 'train_overrides': {}},
    {'name': 'no_dense_dropout', 'model_overrides': {'dropout_rate': 0.0}, 'train_overrides': {}},
    {'name': 'no_conv_dropout', 'model_overrides': {'conv_dropout_rate': 0.0}, 'train_overrides': {}},
    {'name': 'lower_lr', 'model_overrides': {}, 'train_overrides': {'learning_rate': 5e-4}},
    {'name': 'strong_regularization', 'model_overrides': {'l2_regularizer': 2e-3, 'dropout_rate': 0.6, 'conv_dropout_rate': 0.3}, 'train_overrides': {}},
    {'name': 'smaller_model', 'model_overrides': {'conv1_filters': 16, 'conv2_filters': 32, 'dense_units': 64}, 'train_overrides': {}},
]

# Modo recomendado para sua restricao de RAM:
# - 'single': executa exatamente 1 satelite + 1 experimento por vez.
# - 'batch': executa varios em sequencia.
RUN_MODE = 'single'  # 'single' ou 'batch'
SINGLE_SENSOR_KEY = 'sentinel2'
SINGLE_EXPERIMENT_NAME = 'baseline'

# Usado apenas quando RUN_MODE='batch'
EXPERIMENT_NAMES_TO_RUN = None  # ex.: ['baseline', 'no_augmentation']

RUN_EXPERIMENTS = True

assert CODES_PATH.exists(), f'Arquivo nao encontrado: {CODES_PATH}'
assert abs(DATA_CFG['train_size'] + DATA_CFG['val_size'] + DATA_CFG['test_size'] - 1.0) < 1e-8
assert SINGLE_SENSOR_KEY in SENSORS, f'SINGLE_SENSOR_KEY invalido: {SINGLE_SENSOR_KEY}'

print('Output root:', OUTPUT_ROOT)
print('RUN_MODE:', RUN_MODE)
if RUN_MODE == 'single':
    print('Execucao unica:', SINGLE_SENSOR_KEY, '| experimento:', SINGLE_EXPERIMENT_NAME)


## (1) Protocolo Experimental

Critérios e variáveis controladas:
- Divisao fixa por satelite: 70% treino, 15% validacao, 15% teste (estratificado).
- Mesmo split para **todos os experimentos** de um mesmo satelite (comparacao justa).
- Mesma pipeline de preprocessamento por satelite (formato, canais, normalizacao).
- Mesmas metricas de avaliacao em teste:
  - Accuracy, Precision, Recall, F1, Balanced Accuracy, ROC-AUC, PR-AUC.
- Criterio principal de comparacao: **F1 em teste**.
- Criterio secundario: equilibrio entre performance e gap de generalizacao.
- Controle de overfitting: EarlyStopping + checkpoint + ReduceLROnPlateau.


In [ ]:
protocol = {
    'seed': SEED,
    'data_cfg': DATA_CFG,
    'base_model_cfg': BASE_MODEL_CFG,
    'base_train_cfg': BASE_TRAIN_CFG,
    'experiments': [e['name'] for e in EXPERIMENT_PLAN],
    'primary_metric': 'f1',
    'secondary_metrics': ['accuracy', 'balanced_accuracy', 'roc_auc', 'pr_auc'],
    'controlled_variables': [
        'split fixo por satelite',
        'mesmas metricas',
        'mesmo processo de treinamento (callbacks e parada)',
        'mesma estrategia de rotulagem via extracted_codes.json',
    ],
}

protocol_path = OUTPUT_ROOT / 'experimental_protocol.json'
protocol_path.write_text(json.dumps(protocol, ensure_ascii=False, indent=2), encoding='utf-8')
print('Protocolo salvo em:', protocol_path)
protocol


In [ ]:
def to_python(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {k: to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_python(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_python(v) for v in obj]
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def save_json(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(to_python(payload), f, ensure_ascii=False, indent=2)


def load_mlp_baseline_metrics(path: Path) -> dict[str, float | None]:
    if not path.exists():
        return {'accuracy': None, 'loss': None}

    with path.open('r', encoding='utf-8') as f:
        raw = json.load(f)

    acc = raw.get('accuracy')
    if acc is None:
        acc = raw.get('compile_metrics')

    return {
        'accuracy': float(acc) if acc is not None else None,
        'loss': float(raw['loss']) if 'loss' in raw else None,
    }


def band_sort_key(name: str):
    import re
    m = re.match(r'^([A-Za-z]+)(\d+)([A-Za-z]*)$', name)
    if not m:
        return (name, 0, '')
    p, n, s = m.groups()
    return (p, int(n), s)


def infer_band_columns(columns: list[str], prefix: str = 'B') -> list[str]:
    bands = [c for c in columns if c.startswith(prefix) and any(ch.isdigit() for ch in c)]
    return sorted(bands, key=band_sort_key)


def get_parquet_columns(parquet_path: Path) -> list[str]:
    if pq is not None:
        return list(pq.ParquetFile(parquet_path).schema.names)
    return list(pd.read_parquet(parquet_path).columns)


def infer_hw(n_pixels: int) -> tuple[int, int]:
    side = int(math.sqrt(n_pixels))
    if side * side == n_pixels:
        return side, side

    best_h, best_w = 1, n_pixels
    best_gap = abs(best_w - best_h)
    for h in range(1, int(math.sqrt(n_pixels)) + 1):
        if n_pixels % h == 0:
            w = n_pixels // h
            gap = abs(w - h)
            if gap < best_gap:
                best_h, best_w, best_gap = h, w, gap
    return best_h, best_w


def read_rows_for_image_ids(
    parquet_path: Path,
    selected_ids: list[str],
    columns: list[str],
    image_col: str,
    batch_size: int = 200_000,
) -> pd.DataFrame:
    selected_ids = set(map(str, selected_ids))

    if pq is None:
        df = pd.read_parquet(parquet_path, columns=columns)
        return df[df[image_col].astype(str).isin(selected_ids)].copy()

    pf = pq.ParquetFile(parquet_path)
    chunks = []
    for batch in pf.iter_batches(columns=columns, batch_size=batch_size):
        chunk = batch.to_pandas()
        chunk = chunk[chunk[image_col].astype(str).isin(selected_ids)]
        if not chunk.empty:
            chunks.append(chunk)

    if not chunks:
        return pd.DataFrame(columns=columns)
    return pd.concat(chunks, ignore_index=True)


def build_image_tensor_from_long(
    df_rows: pd.DataFrame,
    image_ids: list[str],
    band_cols: list[str],
    target_pixels: int,
    height: int,
    width: int,
    image_col: str,
    pixel_col: str | None,
) -> np.ndarray:
    n_images = len(image_ids)
    n_bands = len(band_cols)

    x = np.zeros((n_images, height, width, n_bands), dtype=np.float32)
    grouped = {str(k): v for k, v in df_rows.groupby(image_col, sort=False)}

    for i, img_id in enumerate(image_ids):
        g = grouped.get(str(img_id))
        if g is None:
            continue

        flat = np.zeros((target_pixels, n_bands), dtype=np.float32)
        vals = g[band_cols].to_numpy(dtype=np.float32, copy=False)

        if pixel_col and pixel_col in g.columns:
            idx = g[pixel_col].to_numpy(dtype=np.int64, copy=False)
            valid = (idx >= 0) & (idx < target_pixels)
            flat[idx[valid]] = vals[valid]
        else:
            n_fill = min(target_pixels, len(vals))
            flat[:n_fill] = vals[:n_fill]

        x[i] = flat.reshape(height, width, n_bands)

    return x


def prepare_sensor_long_dataset(
    sensor_key: str,
    sensor_cfg: dict[str, Any],
    data_cfg: dict[str, Any],
    codes_path: Path,
    seed: int,
) -> dict[str, Any]:
    parquet_path = sensor_cfg['parquet_path']
    sensor_name = sensor_cfg['display_name']

    if not parquet_path.exists():
        raise FileNotFoundError(f'{sensor_name}: parquet nao encontrado em {parquet_path}')

    cols = get_parquet_columns(parquet_path)
    image_col = data_cfg['image_col']
    pixel_col = data_cfg['pixel_col']

    band_cols = infer_band_columns(cols, prefix=data_cfg['band_prefix'])
    if not band_cols:
        raise ValueError(f'{sensor_name}: nenhuma banda detectada.')

    has_pixel_col = pixel_col in cols
    meta_cols = [image_col] + ([pixel_col] if has_pixel_col else [])

    meta_df = pd.read_parquet(parquet_path, columns=meta_cols)
    meta_df[image_col] = meta_df[image_col].astype(str)

    if has_pixel_col:
        pixel_counts = meta_df.groupby(image_col)[pixel_col].nunique().rename('n_pixels')
    else:
        pixel_counts = meta_df.groupby(image_col).size().rename('n_pixels')

    target_pixels = int(pixel_counts.mode().iloc[0])
    valid_ids = pixel_counts[pixel_counts == target_pixels].index.astype(str).tolist()

    max_images = data_cfg['max_images_per_sensor']
    if max_images is not None and len(valid_ids) > max_images:
        rng = np.random.default_rng(seed)
        valid_ids = rng.choice(valid_ids, size=max_images, replace=False).tolist()

    selected_cols = [image_col] + ([pixel_col] if has_pixel_col else []) + band_cols
    rows_df = read_rows_for_image_ids(
        parquet_path=parquet_path,
        selected_ids=valid_ids,
        columns=selected_cols,
        image_col=image_col,
    )
    rows_df[image_col] = rows_df[image_col].astype(str)

    height, width = infer_hw(target_pixels)
    x_all = build_image_tensor_from_long(
        df_rows=rows_df,
        image_ids=valid_ids,
        band_cols=band_cols,
        target_pixels=target_pixels,
        height=height,
        width=width,
        image_col=image_col,
        pixel_col=pixel_col if has_pixel_col else None,
    )

    y_all, _ = labels_from_extracted_codes(valid_ids, codes_path)
    mask = y_all != -1

    x = x_all[mask]
    y = y_all[mask].astype(np.int64)
    image_ids = np.array(valid_ids, dtype=object)[mask]

    counts = pd.Series(y).value_counts().to_dict()
    if 0 not in counts or 1 not in counts:
        raise ValueError(f'{sensor_name}: faltou classe 0/1 apos filtragem.')
    if min(counts.values()) < data_cfg['min_images_per_class']:
        raise ValueError(f'{sensor_name}: poucas imagens por classe {counts}')

    return {
        'X': x,
        'y': y,
        'image_ids': image_ids,
        'band_cols': band_cols,
        'summary': {
            'sensor_key': sensor_key,
            'sensor_name': sensor_name,
            'n_bands': len(band_cols),
            'height': height,
            'width': width,
            'target_pixels': target_pixels,
            'n_images': int(len(y)),
            'class_balance': {str(int(k)): int(v) for k, v in pd.Series(y).value_counts().sort_index().items()},
            'tensor_shape': list(x.shape),
        },
    }


def make_split_indices(y: np.ndarray, train_size: float, val_size: float, test_size: float, seed: int):
    idx = np.arange(len(y))

    train_idx, temp_idx = train_test_split(
        idx,
        test_size=(val_size + test_size),
        stratify=y,
        random_state=seed,
    )

    temp_y = y[temp_idx]
    rel_test = test_size / (val_size + test_size)
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=rel_test,
        stratify=temp_y,
        random_state=seed,
    )

    return {
        'train_idx': train_idx,
        'val_idx': val_idx,
        'test_idx': test_idx,
    }


def subset_by_indices(x: np.ndarray, y: np.ndarray, image_ids: np.ndarray, split_idx: dict[str, np.ndarray]):
    tr = split_idx['train_idx']
    va = split_idx['val_idx']
    te = split_idx['test_idx']
    return {
        'train': (x[tr], y[tr], image_ids[tr]),
        'val': (x[va], y[va], image_ids[va]),
        'test': (x[te], y[te], image_ids[te]),
    }


def to_one_hot(y: np.ndarray, n_classes: int = 2) -> np.ndarray:
    return tf.keras.utils.to_categorical(y.astype(np.int64), num_classes=n_classes)


def merge_cfg(base_cfg: dict[str, Any], overrides: dict[str, Any]) -> dict[str, Any]:
    out = copy.deepcopy(base_cfg)
    for k, v in overrides.items():
        out[k] = v
    return out


def run_single_experiment(
    sensor_key: str,
    sensor_name: str,
    exp_name: str,
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    y_val: np.ndarray,
    x_test: np.ndarray,
    y_test: np.ndarray,
    model_cfg: dict[str, Any],
    train_cfg: dict[str, Any],
    output_root: Path,
    seed: int,
    verbose: int = 1,
) -> dict[str, Any]:
    sensor_dir = output_root / sensor_key
    exp_dir = sensor_dir / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)

    y_train_oh = to_one_hot(y_train, 2)
    y_val_oh = to_one_hot(y_val, 2)
    y_test_oh = to_one_hot(y_test, 2)

    tf.keras.backend.clear_session()

    data_bundle = build_train_val_tf_data(
        x_train=x_train,
        y_train=y_train_oh,
        x_val=x_val,
        y_val=y_val_oh,
        batch_size=train_cfg['batch_size'],
        normalization=train_cfg['normalization'],
        data_format='channels_last',
        target_channels=x_train.shape[-1],
        augment_train=train_cfg['augment_train'],
        seed=seed,
    )

    test_ds, _ = build_tf_data_pipeline(
        x=x_test,
        y=y_test_oh,
        batch_size=train_cfg['batch_size'],
        training=False,
        shuffle=False,
        normalization=train_cfg['normalization'],
        normalizer=data_bundle['normalizer'],
        data_format='channels_last',
        target_channels=x_test.shape[-1],
    )

    model = build_cnn_model(
        input_shape=tuple(data_bundle['train_meta']['input_shape']),
        n_classes=2,
        conv1_filters=model_cfg['conv1_filters'],
        conv2_filters=model_cfg['conv2_filters'],
        kernel_size=tuple(model_cfg['kernel_size']),
        dense_units=model_cfg['dense_units'],
        dropout_rate=model_cfg['dropout_rate'],
        conv_dropout_rate=model_cfg['conv_dropout_rate'],
        l2_regularizer=model_cfg['l2_regularizer'],
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=train_cfg['learning_rate']),
        loss=model.loss,
        metrics=['accuracy'],
    )

    checkpoint_path = exp_dir / 'best_model.keras'
    callbacks = get_training_callbacks(
        checkpoint_path=str(checkpoint_path),
        monitor='val_loss',
        patience=train_cfg['patience'],
        min_delta=train_cfg['min_delta'],
        extra_callbacks=[
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=train_cfg['reduce_lr_factor'],
                patience=train_cfg['reduce_lr_patience'],
                min_lr=train_cfg['min_lr'],
                verbose=1,
            )
        ],
    )

    t0 = time.perf_counter()
    history = model.fit(
        data_bundle['train_ds'],
        validation_data=data_bundle['val_ds'],
        epochs=train_cfg['epochs'],
        callbacks=callbacks,
        verbose=verbose,
    )
    train_time_s = time.perf_counter() - t0

    best_model = tf.keras.models.load_model(checkpoint_path) if checkpoint_path.exists() else model
    eval_dict = best_model.evaluate(test_ds, return_dict=True, verbose=0)

    y_prob = best_model.predict(test_ds, verbose=0)
    y_score = y_prob[:, 1] if y_prob.ndim == 2 else y_prob.ravel()
    y_pred = np.argmax(y_prob, axis=1) if y_prob.ndim == 2 else (y_score >= 0.5).astype(int)

    clf = classification_metrics_extended(y_test, y_pred, y_score)

    train_acc_last = float(history.history['accuracy'][-1]) if history.history.get('accuracy') else None
    val_acc_last = float(history.history['val_accuracy'][-1]) if history.history.get('val_accuracy') else None
    gap = None if (train_acc_last is None or val_acc_last is None) else (train_acc_last - val_acc_last)

    arch = get_model_architecture_summary(model)

    save_json(exp_dir / 'config_model.json', model_cfg)
    save_json(exp_dir / 'config_train.json', train_cfg)
    save_json(exp_dir / 'architecture.json', arch)
    save_json(exp_dir / 'history.json', history.history)

    result = {
        'sensor_key': sensor_key,
        'sensor': sensor_name,
        'experiment': exp_name,
        'epochs_ran': int(len(history.history.get('loss', []))),
        'train_time_s': float(train_time_s),
        'test_loss': eval_dict.get('loss'),
        'test_accuracy_keras': eval_dict.get('accuracy'),
        'accuracy': clf.get('accuracy'),
        'precision': clf.get('precision'),
        'recall': clf.get('recall'),
        'f1': clf.get('f1'),
        'balanced_accuracy': clf.get('balanced_accuracy'),
        'roc_auc': clf.get('roc_auc'),
        'pr_auc': clf.get('pr_auc'),
        'generalization_gap': gap,
    }
    save_json(exp_dir / 'result.json', result)

    # limpeza da execução de experimento
    del y_train_oh, y_val_oh, y_test_oh
    del data_bundle, test_ds, model, best_model, history, y_prob, y_score, y_pred
    tf.keras.backend.clear_session()
    gc.collect()

    return result


def run_sensor_experiment_suite(
    sensor_key: str,
    sensor_cfg: dict[str, Any],
    data_cfg: dict[str, Any],
    base_model_cfg: dict[str, Any],
    base_train_cfg: dict[str, Any],
    experiment_plan: list[dict[str, Any]],
    output_root: Path,
    codes_path: Path,
    seed: int,
    experiment_names_to_run: list[str] | None = None,
    verbose: int = 1,
) -> list[dict[str, Any]]:
    sensor_name = sensor_cfg['display_name']
    print(f'\n========== {sensor_name} ==========' )
    print('Carregando dataset do sensor...')

    prep = prepare_sensor_long_dataset(
        sensor_key=sensor_key,
        sensor_cfg=sensor_cfg,
        data_cfg=data_cfg,
        codes_path=codes_path,
        seed=seed,
    )

    sensor_dir = output_root / sensor_key
    sensor_dir.mkdir(parents=True, exist_ok=True)
    save_json(sensor_dir / 'data_summary.json', prep['summary'])

    split_idx = make_split_indices(
        y=prep['y'],
        train_size=data_cfg['train_size'],
        val_size=data_cfg['val_size'],
        test_size=data_cfg['test_size'],
        seed=seed,
    )
    save_json(sensor_dir / 'split_indices_summary.json', {
        'n_train': int(len(split_idx['train_idx'])),
        'n_val': int(len(split_idx['val_idx'])),
        'n_test': int(len(split_idx['test_idx'])),
    })

    subsets = subset_by_indices(prep['X'], prep['y'], prep['image_ids'], split_idx)
    x_train, y_train, _ = subsets['train']
    x_val, y_val, _ = subsets['val']
    x_test, y_test, _ = subsets['test']

    results = []
    for exp in experiment_plan:
        exp_name = exp['name']
        if experiment_names_to_run is not None and exp_name not in experiment_names_to_run:
            continue

        model_cfg = merge_cfg(base_model_cfg, exp.get('model_overrides', {}))
        train_cfg = merge_cfg(base_train_cfg, exp.get('train_overrides', {}))

        print(f'\n[{sensor_name}] Experimento: {exp_name}')
        res = run_single_experiment(
            sensor_key=sensor_key,
            sensor_name=sensor_name,
            exp_name=exp_name,
            x_train=x_train,
            y_train=y_train,
            x_val=x_val,
            y_val=y_val,
            x_test=x_test,
            y_test=y_test,
            model_cfg=model_cfg,
            train_cfg=train_cfg,
            output_root=output_root,
            seed=seed,
            verbose=verbose,
        )
        results.append(res)

    # limpeza do sensor inteiro
    del prep, split_idx, subsets
    del x_train, y_train, x_val, y_val, x_test, y_test
    gc.collect()
    tf.keras.backend.clear_session()

    return results


## (2) Execucao de multiplos experimentos

A execucao eh sequencial por satelite (um por vez), e dentro de cada satelite
rodamos o conjunto de configuracoes definido em `EXPERIMENT_PLAN`.


In [ ]:
run_results: list[dict[str, Any]] = []

if RUN_EXPERIMENTS:
    if RUN_MODE == 'single':
        sensor_keys = [SINGLE_SENSOR_KEY]
        names_for_run = [SINGLE_EXPERIMENT_NAME]
    elif RUN_MODE == 'batch':
        sensor_keys = SENSOR_ORDER
        names_for_run = EXPERIMENT_NAMES_TO_RUN
    else:
        raise ValueError("RUN_MODE deve ser 'single' ou 'batch'.")

    for sensor_key in sensor_keys:
        sensor_cfg = SENSORS[sensor_key]
        if not sensor_cfg['parquet_path'].exists():
            print(f"[SKIP] {sensor_cfg['display_name']}: parquet ausente em {sensor_cfg['parquet_path']}")
            continue

        sensor_results = run_sensor_experiment_suite(
            sensor_key=sensor_key,
            sensor_cfg=sensor_cfg,
            data_cfg=DATA_CFG,
            base_model_cfg=BASE_MODEL_CFG,
            base_train_cfg=BASE_TRAIN_CFG,
            experiment_plan=EXPERIMENT_PLAN,
            output_root=OUTPUT_ROOT,
            codes_path=CODES_PATH,
            seed=SEED,
            experiment_names_to_run=names_for_run,
            verbose=1,
        )
        run_results.extend(sensor_results)
else:
    print('RUN_EXPERIMENTS=False, execucao pulada.')

run_df = pd.DataFrame(run_results)
store_path = OUTPUT_ROOT / 'all_experiment_results.csv'

if store_path.exists():
    stored_df = pd.read_csv(store_path)
else:
    stored_df = pd.DataFrame()

if run_df.empty:
    results_df = stored_df.copy()
else:
    combined = pd.concat([stored_df, run_df], ignore_index=True)
    if not combined.empty and {'sensor_key', 'experiment'}.issubset(combined.columns):
        combined = combined.drop_duplicates(subset=['sensor_key', 'experiment'], keep='last')
    combined.to_csv(store_path, index=False)
    results_df = combined

print('Resultados desta execucao:', len(run_df))
print('Resultados acumulados:', len(results_df))
if not run_df.empty:
    display(run_df)
else:
    print('Nenhum resultado novo foi produzido neste run.')


## (3) Analise comparativa de desempenho

Consolidacao com tabelas e graficos comparativos por sensor e por experimento.


In [ ]:
if results_df.empty:
    print('Sem resultados para comparar.')
else:
    # Ranking por F1
    ranking_f1 = (
        results_df
        .sort_values(['sensor', 'f1'], ascending=[True, False])
        .groupby('sensor', as_index=False)
        .head(5)
    )
    display(ranking_f1[['sensor', 'experiment', 'f1', 'accuracy', 'roc_auc', 'pr_auc']])

    plt.figure(figsize=(12, 5))
    sns.barplot(data=results_df, x='sensor', y='f1', hue='experiment')
    plt.title('F1 por experimento e por satelite')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 5))
    sns.barplot(data=results_df, x='sensor', y='accuracy', hue='experiment')
    plt.title('Accuracy por experimento e por satelite')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

    pivot_f1 = results_df.pivot_table(index='experiment', columns='sensor', values='f1', aggfunc='mean')
    display(pivot_f1)

    (OUTPUT_ROOT / 'comparison').mkdir(parents=True, exist_ok=True)
    pivot_f1.to_csv(OUTPUT_ROOT / 'comparison' / 'pivot_f1.csv')


## (4) Estudo de impacto / ablacao

Ablacoes avaliadas (comparadas ao baseline):
- `no_augmentation`
- `no_l2`
- `no_dense_dropout`
- `no_conv_dropout`


In [ ]:
ABLATION_NAMES = ['no_augmentation', 'no_l2', 'no_dense_dropout', 'no_conv_dropout']

if results_df.empty:
    ablation_df = pd.DataFrame()
    print('Sem resultados para ablacao.')
else:
    baseline = results_df[results_df['experiment'] == 'baseline'][['sensor', 'f1', 'accuracy', 'roc_auc', 'pr_auc']].copy()
    baseline = baseline.rename(columns={
        'f1': 'f1_baseline',
        'accuracy': 'accuracy_baseline',
        'roc_auc': 'roc_auc_baseline',
        'pr_auc': 'pr_auc_baseline',
    })

    ablation_rows = []
    for name in ABLATION_NAMES:
        cur = results_df[results_df['experiment'] == name][['sensor', 'experiment', 'f1', 'accuracy', 'roc_auc', 'pr_auc']].copy()
        if cur.empty:
            continue
        merged = cur.merge(baseline, on='sensor', how='left')
        merged['delta_f1'] = merged['f1'] - merged['f1_baseline']
        merged['delta_accuracy'] = merged['accuracy'] - merged['accuracy_baseline']
        merged['delta_roc_auc'] = merged['roc_auc'] - merged['roc_auc_baseline']
        merged['delta_pr_auc'] = merged['pr_auc'] - merged['pr_auc_baseline']
        ablation_rows.append(merged)

    ablation_df = pd.concat(ablation_rows, ignore_index=True) if ablation_rows else pd.DataFrame()
    display(ablation_df)

    if not ablation_df.empty:
        plt.figure(figsize=(12, 5))
        sns.barplot(data=ablation_df, x='sensor', y='delta_f1', hue='experiment')
        plt.axhline(0, color='black', linewidth=1)
        plt.title('Impacto de ablacao em F1 (delta vs baseline)')
        plt.tight_layout()
        plt.show()

        ablation_df.to_csv(OUTPUT_ROOT / 'ablation_results.csv', index=False)


## (5) Interpretacao critica dos achados

A analise textual abaixo e obrigatoria no artefato e sintetiza:
- conclusoes principais;
- limitacoes;
- recomendacoes para a proxima Sprint.


In [ ]:
analysis_lines = []

mlp_baseline = load_mlp_baseline_metrics(MLP_BASELINE_PATH)

if results_df.empty:
    analysis_lines.append('Nenhum experimento foi executado, entao nao ha conclusoes quantitativas.')
else:
    # Melhor experimento por satelite
    best_by_sensor = results_df.sort_values('f1', ascending=False).groupby('sensor', as_index=False).first()
    for _, row in best_by_sensor.iterrows():
        analysis_lines.append(
            f"{row['sensor']}: melhor configuracao foi {row['experiment']} "
            f"(F1={row['f1']:.4f}, Accuracy={row['accuracy']:.4f}, ROC-AUC={row['roc_auc']:.4f})."
        )

    # Tendencia global
    global_mean = results_df.groupby('experiment', as_index=False)['f1'].mean().sort_values('f1', ascending=False)
    top_exp = global_mean.iloc[0]
    analysis_lines.append(
        f"Globalmente, {top_exp['experiment']} teve melhor media de F1 entre sensores ({top_exp['f1']:.4f})."
    )

    # Efeito da augmentacao
    if {'baseline', 'no_augmentation'}.issubset(set(results_df['experiment'])):
        b = results_df[results_df['experiment'] == 'baseline'][['sensor', 'f1']].rename(columns={'f1': 'f1_baseline'})
        n = results_df[results_df['experiment'] == 'no_augmentation'][['sensor', 'f1']].rename(columns={'f1': 'f1_no_aug'})
        m = b.merge(n, on='sensor', how='inner')
        if not m.empty:
            delta = (m['f1_baseline'] - m['f1_no_aug']).mean()
            if delta > 0:
                analysis_lines.append(
                    f"Em media, usar augmentation melhorou F1 em {delta:+.4f} vs no_augmentation."
                )
            else:
                analysis_lines.append(
                    f"Em media, augmentation nao trouxe ganho de F1 (delta medio {delta:+.4f})."
                )

    # Comparacao com MLP sprint 2 (limitada)
    if mlp_baseline.get('accuracy') is not None:
        best_acc = results_df['accuracy'].max()
        analysis_lines.append(
            f"Melhor accuracy de CNN nos experimentos: {best_acc:.4f}; "
            f"baseline MLP Sprint 2 registrado: {mlp_baseline['accuracy']:.4f}."
        )

    # Limitacoes metodologicas
    analysis_lines.append(
        'Limitacao: o baseline MLP salvo no repositorio tem poucas metricas detalhadas, '
        'entao a comparacao com CNN fica parcialmente restrita.'
    )
    analysis_lines.append(
        'Limitacao: diferencas de resolucao/espectro entre sensores podem exigir '
        'arquiteturas especificas por sensor para desempenho maximo.'
    )
    analysis_lines.append(
        'Recomendacao Sprint seguinte: fixar 1-2 melhores configuracoes por sensor e '
        'rodar validacao mais robusta (ex.: repeticoes com seeds diferentes ou CV por grupos de image_id).'
    )

for i, line in enumerate(analysis_lines, start=1):
    print(f'{i}. {line}')

analysis_path = OUTPUT_ROOT / 'critical_analysis.txt'
analysis_path.write_text('\n'.join(analysis_lines), encoding='utf-8')

summary_payload = {
    'n_results': int(len(results_df)),
    'sensors_with_results': sorted(results_df['sensor'].unique().tolist()) if not results_df.empty else [],
    'mlp_baseline': mlp_baseline,
    'critical_analysis_path': str(analysis_path),
}
save_json(OUTPUT_ROOT / 'run_summary.json', summary_payload)

print('\nArquivos finais:')
print('-', OUTPUT_ROOT / 'experimental_protocol.json')
if (OUTPUT_ROOT / 'all_experiment_results.csv').exists():
    print('-', OUTPUT_ROOT / 'all_experiment_results.csv')
if (OUTPUT_ROOT / 'ablation_results.csv').exists():
    print('-', OUTPUT_ROOT / 'ablation_results.csv')
print('-', analysis_path)
print('-', OUTPUT_ROOT / 'run_summary.json')
